# Inspect Precomputed CDR Metrics

This notebook checks parquet files generated by `preprocess/precompute_decoy_metrics.py` under `cdr-data/metrics_precomputed`.

It focuses on quick validation:
- which source files exist
- row counts and columns
- identity key uniqueness: `target_id + source + seed + sample`
- target-level CDR counts
- metric ranges, NaN rates, and success rates
- suspicious rows such as high missing backbone atom counts

In [ ]:
from pathlib import Path
import math
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/cdr_scoring_matplotlib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

METRICS_ROOT = Path("/home/sujin/projects/cdr-scoring/cdr-data/metrics_precomputed")
METRICS_ROOT

In [ ]:
parquet_files = sorted(METRICS_ROOT.glob("*/*/*.parquet"))
files_df = pd.DataFrame({
    "source": [p.parts[-3] for p in parquet_files],
    "group": [p.parts[-2] for p in parquet_files],
    "file": [p.name for p in parquet_files],
    "path": [str(p) for p in parquet_files],
    "size_mb": [p.stat().st_size / 1024**2 for p in parquet_files],
})
display(files_df.sort_values(["source", "group", "file"]))

if files_df.empty:
    raise FileNotFoundError(f"No parquet files found under {METRICS_ROOT}")

In [ ]:
def read_source_table(source: str, group: str, filename: str) -> pd.DataFrame:
    path = METRICS_ROOT / source / group / filename
    if not path.exists():
        return pd.DataFrame()
    return pd.read_parquet(path)

sources = sorted(files_df["source"].unique())
tables = {}
for source in sources:
    tables[(source, "targets")] = read_source_table(source, "targets", "target_metrics.parquet")
    tables[(source, "loop")] = read_source_table(source, "metrics", "loop_metrics.parquet")
    tables[(source, "interface")] = read_source_table(source, "metrics", "interface_metrics.parquet")
    tables[(source, "dockq")] = read_source_table(source, "metrics", "dockq_metrics.parquet")

overview_rows = []
for (source, table_name), df in tables.items():
    if df.empty:
        continue
    overview_rows.append({
        "source": source,
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "targets": df["target_id"].nunique() if "target_id" in df else np.nan,
        "decoys": len(df[["target_id", "source", "seed", "sample"]].drop_duplicates()) if {"target_id", "source", "seed", "sample"}.issubset(df.columns) else np.nan,
    })

overview = pd.DataFrame(overview_rows).sort_values(["source", "table"])
display(overview)

## Inspect One Source

Change `SOURCE` below to inspect a specific source in detail.

In [ ]:
SOURCE = sources[0]
SOURCE

In [ ]:
target_df = tables[(SOURCE, "targets")]
loop_df = tables[(SOURCE, "loop")]

print("target_metrics columns:")
display(pd.DataFrame({"column": target_df.columns, "dtype": [str(t) for t in target_df.dtypes]}))
display(target_df.head(10))

print("loop_metrics columns:")
display(pd.DataFrame({"column": loop_df.columns, "dtype": [str(t) for t in loop_df.dtypes]}))
display(loop_df.head(10))

## Key Checks

The expected decoy lookup key is `target_id + source + seed + sample`.

In [ ]:
KEY = ["target_id", "source", "seed", "sample"]

key_check_rows = []
duplicate_examples = {}
for source in sources:
    df = tables[(source, "loop")]
    if df.empty:
        continue
    missing_key_cols = [c for c in KEY if c not in df.columns]
    if missing_key_cols:
        key_check_rows.append({"source": source, "rows": len(df), "missing_key_cols": missing_key_cols})
        continue
    dup_mask = df.duplicated(KEY, keep=False)
    key_check_rows.append({
        "source": source,
        "rows": len(df),
        "unique_keys": len(df[KEY].drop_duplicates()),
        "duplicate_key_rows": int(dup_mask.sum()),
        "missing_seed_rows": int(df["seed"].isna().sum()),
        "missing_sample_rows": int(df["sample"].isna().sum()),
    })
    if dup_mask.any():
        duplicate_examples[source] = df.loc[dup_mask, KEY + ["decoy_id", "decoy_path"]].head(20)

display(pd.DataFrame(key_check_rows))
for source, dup_df in duplicate_examples.items():
    print(f"Duplicate examples: {source}")
    display(dup_df)

## Target-Level CDR Counts

In [ ]:
count_cols = ["H1_count", "H2_count", "H3_count", "L1_count", "L2_count", "L3_count"]
target_summary_rows = []
for source in sources:
    df = tables[(source, "targets")]
    if df.empty:
        continue
    row = {"source": source, "targets": len(df), "unique_targets": df["target_id"].nunique()}
    for col in count_cols:
        if col in df:
            row[f"{col}_min"] = df[col].min()
            row[f"{col}_median"] = df[col].median()
            row[f"{col}_max"] = df[col].max()
            row[f"{col}_zero"] = int((df[col] == 0).sum())
    target_summary_rows.append(row)

target_summary = pd.DataFrame(target_summary_rows)
display(target_summary)

if not target_df.empty:
    display(target_df[["target_id", "source", "has_holo_antigen", "antibody_chains", "antigen_chains"] + count_cols].head(20))

## Loop Metric Sanity Summary

In [ ]:
metric_cols = [
    "global_loop_rmsd", "global_loop_lddt",
    "H1_loop_rmsd", "H2_loop_rmsd", "H3_loop_rmsd", "L1_loop_rmsd", "L2_loop_rmsd", "L3_loop_rmsd",
    "H1_loop_lddt", "H2_loop_lddt", "H3_loop_lddt", "L1_loop_lddt", "L2_loop_lddt", "L3_loop_lddt",
]

summary_rows = []
for source in sources:
    df = tables[(source, "loop")]
    if df.empty:
        continue
    row = {"source": source, "rows": len(df), "targets": df["target_id"].nunique()}
    for col in ["global_loop_rmsd", "global_loop_lddt"]:
        values = pd.to_numeric(df[col], errors="coerce") if col in df else pd.Series(dtype=float)
        finite = values[np.isfinite(values)]
        row[f"{col}_finite"] = int(finite.size)
        row[f"{col}_nan"] = int(values.isna().sum())
        row[f"{col}_min"] = finite.min() if finite.size else np.nan
        row[f"{col}_median"] = finite.median() if finite.size else np.nan
        row[f"{col}_max"] = finite.max() if finite.size else np.nan
    if "global_loop_rmsd" in df:
        rmsd = pd.to_numeric(df["global_loop_rmsd"], errors="coerce")
        row["success_rmsd_le_2"] = float((rmsd <= 2.0).mean())
    if "global_loop_lddt" in df:
        lddt = pd.to_numeric(df["global_loop_lddt"], errors="coerce")
        row["success_lddt_ge_0p8"] = float((lddt >= 0.8).mean())
    if "missing_backbone_atom_count" in df:
        row["missing_backbone_ge_5"] = int((df["missing_backbone_atom_count"] >= 5).sum())
    summary_rows.append(row)

loop_summary = pd.DataFrame(summary_rows)
display(loop_summary)

In [ ]:
if not loop_df.empty:
    print("Rows with missing backbone atom count >= 5")
    cols = ["target_id", "source", "seed", "sample", "decoy_id", "missing_backbone_atom_count", "missing_backbone_report", "decoy_path"]
    display(loop_df.loc[loop_df["missing_backbone_atom_count"] >= 5, cols].head(50))

    print("Rows with NaN global loop RMSD or lDDT")
    nan_mask = loop_df["global_loop_rmsd"].isna() | loop_df["global_loop_lddt"].isna()
    display(loop_df.loc[nan_mask, ["target_id", "source", "seed", "sample", "decoy_id", "global_loop_rmsd", "global_loop_lddt", "missing_backbone_report", "decoy_path"]].head(50))

## Metric Distributions

In [ ]:
all_loop = []
for source in sources:
    df = tables[(source, "loop")]
    if not df.empty:
        all_loop.append(df.assign(source=source))
all_loop = pd.concat(all_loop, ignore_index=True) if all_loop else pd.DataFrame()

if not all_loop.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for source, sub in all_loop.groupby("source"):
        axes[0].hist(sub["global_loop_rmsd"].dropna(), bins=50, alpha=0.45, label=source)
        axes[1].hist(sub["global_loop_lddt"].dropna(), bins=50, alpha=0.45, label=source)
    axes[0].set_title("global_loop_rmsd")
    axes[0].set_xlabel("A")
    axes[0].set_ylabel("count")
    axes[1].set_title("global_loop_lddt")
    axes[1].set_xlabel("lDDT")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No loop metrics loaded")

In [ ]:
if not all_loop.empty:
    per_cdr_rmsd = ["H1_loop_rmsd", "H2_loop_rmsd", "H3_loop_rmsd", "L1_loop_rmsd", "L2_loop_rmsd", "L3_loop_rmsd"]
    per_cdr_lddt = ["H1_loop_lddt", "H2_loop_lddt", "H3_loop_lddt", "L1_loop_lddt", "L2_loop_lddt", "L3_loop_lddt"]
    display(all_loop.groupby("source")[per_cdr_rmsd + per_cdr_lddt].agg(["count", "median", "min", "max"]))

## Inspect One Target

In [ ]:
if not loop_df.empty:
    TARGET_ID = loop_df["target_id"].iloc[0]
    sub = loop_df[loop_df["target_id"] == TARGET_ID].copy()
    print("SOURCE =", SOURCE, "TARGET_ID =", TARGET_ID, "n =", len(sub))
    display(target_df[target_df["target_id"] == TARGET_ID])
    display(sub.sort_values("global_loop_rmsd").head(20))

## Boltz2 Apo vs Holo Analysis

The following cells focus on Boltz2 and split decoys by the target-level `has_holo_antigen` flag from `targets/target_metrics.parquet`.

In [ ]:
boltz_source = next((s for s in sources if s.lower() == "boltz2"), None)
if boltz_source is None:
    raise ValueError("Boltz2 source was not found under METRICS_ROOT")

boltz_loop = tables[(boltz_source, "loop")].copy()
boltz_targets = tables[(boltz_source, "targets")].copy()

target_state_cols = ["target_id", "source", "has_holo_antigen", "antigen_chains"]
boltz = boltz_loop.merge(
    boltz_targets[target_state_cols].drop_duplicates(),
    on=["target_id", "source"],
    how="left",
)
boltz["target_state"] = np.where(boltz["has_holo_antigen"].fillna(False), "holo", "apo")

print("Boltz2 rows:", len(boltz), "targets:", boltz["target_id"].nunique())
display(boltz.groupby("target_state").agg(rows=("target_id", "size"), targets=("target_id", "nunique")))
display(boltz.head())

### Figure 1. Boltz2 global_loop_rmsd / global_loop_lddt distribution, apo vs holo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = {"apo": "tab:blue", "holo": "tab:orange"}

for state, sub in boltz.groupby("target_state"):
    axes[0].hist(sub["global_loop_rmsd"].dropna(), bins=40, alpha=0.55, label=f"{state} (n={len(sub)})", color=colors.get(state))
    axes[1].hist(sub["global_loop_lddt"].dropna(), bins=40, alpha=0.55, label=f"{state} (n={len(sub)})", color=colors.get(state))

axes[0].set_title("Boltz2 global_loop_rmsd")
axes[0].set_xlabel("RMSD (A)")
axes[0].set_ylabel("Decoy count")
axes[0].legend()

axes[1].set_title("Boltz2 global_loop_lddt")
axes[1].set_xlabel("lDDT")
axes[1].set_ylabel("Decoy count")
axes[1].legend()

plt.tight_layout()
plt.show()

display(boltz.groupby("target_state")[["global_loop_rmsd", "global_loop_lddt"]].agg(["count", "median", "mean", "min", "max"]))

### Figure 2. Per-CDR loop_rmsd boxplot: H1-H3/L1-L3, apo vs holo

In [ ]:
cdr_order = ["H1", "H2", "H3", "L1", "L2", "L3"]
rmsd_cols = [f"{cdr}_loop_rmsd" for cdr in cdr_order]

box_data = []
positions = []
box_colors = []
labels = []
pos = 1
for cdr in cdr_order:
    for offset, state in enumerate(["apo", "holo"]):
        values = boltz.loc[boltz["target_state"] == state, f"{cdr}_loop_rmsd"].dropna().to_numpy()
        box_data.append(values)
        positions.append(pos + offset * 0.35)
        box_colors.append(colors.get(state, "gray"))
    labels.append(cdr)
    pos += 1.1

fig, ax = plt.subplots(figsize=(11, 4.5))
bp = ax.boxplot(box_data, positions=positions, widths=0.28, patch_artist=True, showfliers=False)
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.55)

centers = [1 + i * 1.1 + 0.175 for i in range(len(cdr_order))]
ax.set_xticks(centers)
ax.set_xticklabels(labels)
ax.set_ylabel("loop_rmsd (A)")
ax.set_title("Boltz2 per-CDR loop_rmsd: apo vs holo")
ax.grid(axis="y", alpha=0.25)

handles = [plt.Rectangle((0, 0), 1, 1, color=colors[state], alpha=0.55) for state in ["apo", "holo"]]
ax.legend(handles, ["apo", "holo"], title="target state")
plt.tight_layout()
plt.show()

per_cdr_summary = boltz.groupby("target_state")[rmsd_cols].agg(["count", "median", "mean"])
display(per_cdr_summary)

### Figure 3. Boltz2 ranking vs global_loop_lddt scatter + Spearman summary

In [ ]:
scatter_df = boltz[["target_state", "target_id", "seed", "sample", "ranking", "global_loop_lddt"]].copy()
scatter_df["ranking"] = pd.to_numeric(scatter_df["ranking"], errors="coerce")
scatter_df["global_loop_lddt"] = pd.to_numeric(scatter_df["global_loop_lddt"], errors="coerce")
scatter_df = scatter_df.dropna(subset=["ranking", "global_loop_lddt"])

def spearman_summary(df, label):
    if len(df) < 2 or df["ranking"].nunique() < 2 or df["global_loop_lddt"].nunique() < 2:
        return {"group": label, "n": len(df), "rho": np.nan, "pvalue": np.nan}
    try:
        from scipy.stats import spearmanr
        rho, pvalue = spearmanr(df["ranking"], df["global_loop_lddt"], nan_policy="omit")
    except Exception:
        rho = df[["ranking", "global_loop_lddt"]].corr(method="spearman").iloc[0, 1]
        pvalue = np.nan
    return {"group": label, "n": len(df), "rho": float(rho), "pvalue": float(pvalue) if pd.notna(pvalue) else np.nan}

summary_rows = [spearman_summary(scatter_df, "all")]
for state, sub in scatter_df.groupby("target_state"):
    summary_rows.append(spearman_summary(sub, state))
spearman_df = pd.DataFrame(summary_rows)
display(spearman_df)

fig, ax = plt.subplots(figsize=(7, 5))
for state, sub in scatter_df.groupby("target_state"):
    ax.scatter(sub["ranking"], sub["global_loop_lddt"], s=28, alpha=0.7, label=f"{state} (n={len(sub)})", color=colors.get(state))

rho_all = spearman_df.loc[spearman_df["group"] == "all", "rho"].iloc[0]
ax.set_title(f"Boltz2 ranking vs global_loop_lddt (Spearman rho={rho_all:.3f})")
ax.set_xlabel("Boltz2 ranking")
ax.set_ylabel("global_loop_lddt")
ax.grid(alpha=0.25)
ax.legend(title="target state")
plt.tight_layout()
plt.show()

display(scatter_df.sort_values(["target_id", "ranking"]).head(20))